In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# =========================
# PATHS
# =========================
BASE_DIR = Path(r"C:\Plegma_Programming")
INPUT_DIR = BASE_DIR / "Processed_Houses_Final"
OUTPUT_DIR = BASE_DIR / "Processed_Houses_Hourly"
MASTER_OUTPUT_FILE = BASE_DIR / "PLEGMA_hourly_master_dataset.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# HOUSES TO KEEP
# =========================
VALID_HOUSES = [
    "House_01", "House_02", "House_03", "House_04", "House_05",
    "House_07", "House_09", "House_10", "House_11", "House_13"
]

# =========================
# GREEK HOLIDAYS
# =========================
GREEK_HOLIDAYS = {
    # 2022
    "2022-08-15",
    "2022-10-28",
    "2022-12-25",
    "2022-12-26",

    # 2023
    "2023-01-01",
    "2023-01-06",
    "2023-02-27",
    "2023-03-25",
    "2023-04-14",
    "2023-04-16",
    "2023-04-17",
    "2023-05-01",
    "2023-06-05",
    "2023-08-15",
}

def get_season(month: int) -> str:
    if month in [12, 1, 2]:
        return "winter"
    elif month in [3, 4, 5]:
        return "spring"
    elif month in [6, 7, 8]:
        return "summer"
    else:
        return "autumn"

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["hour"] = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.dayofweek
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
    df["month"] = df["timestamp"].dt.month
    df["season"] = df["month"].apply(get_season)

    df["date_only"] = df["timestamp"].dt.strftime("%Y-%m-%d")
    df["is_holiday"] = df["date_only"].isin(GREEK_HOLIDAYS).astype(int)
    df.drop(columns=["date_only"], inplace=True)

    return df

def convert_halfhour_to_hourly(df: pd.DataFrame) -> pd.DataFrame:
    """
    Input:
      30-min rows with timestamps like:
      00:00, 00:30, 01:00, 01:30, ...

    Output:
      hourly rows using end-of-hour timestamp:
      01:00 = aggregation of [00:30, 01:00]
      02:00 = aggregation of [01:30, 02:00]
      etc.
    """
    df = df.copy()

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"]).sort_values("timestamp")

    # Shift by +30 minutes to align half-hour rows to end-of-hour aggregation
    df["timestamp_shifted"] = df["timestamp"] + pd.Timedelta(minutes=30)
    df = df.set_index("timestamp_shifted")

    agg_dict = {
        "energy_Wh": "sum",
        "internal_temperature": "mean",
        "internal_humidity": "mean",
        "external_temperature": "mean",
        "external_humidity": "mean"
    }

    hourly = df.resample("1h", label="right", closed="right").agg(agg_dict)
    hourly = hourly.reset_index().rename(columns={"timestamp_shifted": "timestamp"})

    # Rename to match IDEAL-style naming where possible
    hourly = hourly.rename(columns={"energy_Wh": "consumption_Wh"})

    return hourly

all_hourly_dfs = []
processed_summary = []

for house in VALID_HOUSES:
    file_path = INPUT_DIR / f"{house}.csv"

    if not file_path.exists():
        print(f"[WARNING] File not found: {file_path}")
        continue

    print(f"Processing {house} ...")

    df = pd.read_csv(file_path)
    df.columns = [c.strip() for c in df.columns]

    expected_cols = [
        "timestamp",
        "energy_Wh",
        "internal_temperature",
        "internal_humidity",
        "external_temperature",
        "external_humidity"
    ]

    missing_cols = [c for c in expected_cols if c not in df.columns]
    if missing_cols:
        print(f"[WARNING] Missing columns in {house}: {missing_cols}")
        continue

    df = df[expected_cols].copy()

    hourly_df = convert_halfhour_to_hourly(df)
    hourly_df.insert(0, "home_id", house)
    hourly_df = add_time_features(hourly_df)

    # Reorder columns close to IDEAL_final_hourly_dataset
    hourly_df = hourly_df[
        [
            "home_id",
            "timestamp",
            "consumption_Wh",
            "internal_temperature",
            "external_temperature",
            "internal_humidity",
            "external_humidity",
            "hour",
            "day_of_week",
            "is_weekend",
            "is_holiday",
            "month",
            "season",
        ]
    ]

    out_file = OUTPUT_DIR / f"{house}_hourly.csv"
    hourly_df.to_csv(out_file, index=False)

    all_hourly_dfs.append(hourly_df)

    processed_summary.append({
        "house_id": house,
        "input_rows_30min": len(df),
        "output_rows_hourly": len(hourly_df),
        "start_timestamp": hourly_df["timestamp"].min(),
        "end_timestamp": hourly_df["timestamp"].max(),
        "output_file": str(out_file)
    })

if not all_hourly_dfs:
    raise ValueError("No hourly datasets were created. Check paths/files.")

summary_df = pd.DataFrame(processed_summary)
print("\nProcessing summary:")
print(summary_df)

plegma_hourly_master = pd.concat(all_hourly_dfs, ignore_index=True)
plegma_hourly_master = plegma_hourly_master.sort_values(["home_id", "timestamp"]).reset_index(drop=True)

plegma_hourly_master.to_csv(MASTER_OUTPUT_FILE, index=False)

print("\nMaster dataset saved to:")
print(MASTER_OUTPUT_FILE)

print("\nShape:")
print(plegma_hourly_master.shape)

print("\nColumns:")
print(list(plegma_hourly_master.columns))

print("\nFirst rows:")
print(plegma_hourly_master.head(20))

Processing House_01 ...
Processing House_02 ...
Processing House_03 ...
Processing House_04 ...
Processing House_05 ...
Processing House_07 ...
Processing House_09 ...
Processing House_10 ...
Processing House_11 ...
Processing House_13 ...

Processing summary:
   house_id  input_rows_30min  output_rows_hourly     start_timestamp  \
0  House_01             21265               10633 2022-07-15 01:00:00   
1  House_02              9157                4579 2022-09-17 01:00:00   
2  House_03             21431               10716 2022-07-11 14:00:00   
3  House_04             17059                8530 2022-10-10 15:00:00   
4  House_05             13103                6552 2023-01-01 01:00:00   
5  House_07             16033                8017 2022-11-01 01:00:00   
6  House_09             11616                5809 2023-02-01 01:00:00   
7  House_10              7807                3904 2023-04-21 09:00:00   
8  House_11              7863                3932 2023-04-20 06:00:00   
9  House_